In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import sklearn
import os
import yfinance as yf
import pandas_datareader.data as web
import time
import requests
import plotly.express as px  # Added for interactive hover
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from google.colab import drive
from datetime import datetime
from scipy.stats import norm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Setup Standard Directory
base_dir = '/content/drive/MyDrive/MSFin_Python'
sub_folders = ['data', 'notebooks', 'output']

for folder in sub_folders:
    os.makedirs(os.path.join(base_dir, folder), exist_ok=True)

print(f"Environment Ready: Drive mounted and folders verified at {base_dir}")


In [ ]:
# --- CSV FILE NAME FOR THIS ANALYSIS ---
SOURCE_FILENAME = 'L11_Path_114_x250_GARCH_FWD.csv'

# --- AUTOMATED PATH BUILDING ---
base_name = os.path.splitext(SOURCE_FILENAME)[0]
data_dir = os.path.join(base_dir, 'data')

# Define all file addresses based on the source
input_file = os.path.join(data_dir, SOURCE_FILENAME)
output_file = os.path.join(data_dir, f'L11_DD_Analysis_Results_{base_name}.csv')

print(f"Processing File: {input_file}")
print(f"Results will save to: {os.path.basename(output_file)}")

In [ ]:
def load_path_data(file_path):
    print(f"Loading data from: {file_path}")
    try:
        df = pd.read_csv(file_path)
        # Ensure Month_Num is the index so it's not treated as a return path
        if 'Month_Num' in df.columns:
            df = df.set_index('Month_Num')
        return df
    except FileNotFoundError:
        print("ERROR: L11_Paths.csv not found in /MSFin_Python/data/")
        return None

df_paths = load_path_data(input_file)

In [ ]:
def calculate_path_metrics(df):
    results = []

    for col in df.columns:
        returns = df[col]

        # 1. Annualized Volatility (Std Dev * sqrt(12))
        ann_vol = returns.std() * np.sqrt(12)

        # 2. Max Drawdown
        # Create wealth index starting at 1.0
        wealth_index = (1 + returns).cumprod()
        previous_peaks = wealth_index.cummax()
        drawdowns = (wealth_index - previous_peaks) / previous_peaks
        max_dd = drawdowns.min()

        # 3. Cumulative Return (Ending Value - 1)
        cum_ret = wealth_index.iloc[-1] - 1

        results.append({
            'Path': col,
            'Ann_Volatility': ann_vol,
            'Max_Drawdown': max_dd,
            'Cumulative_Return': cum_ret
        })

    return pd.DataFrame(results)

if df_paths is not None:
    df_results = calculate_path_metrics(df_paths)
    print("Metrics Calculation Complete.")

In [ ]:
def export_analysis(df, path):
    df.to_csv(path, index=False)
    print(f"Analysis saved to: {path}")

if 'df_results' in locals():
    export_analysis(df_results, output_file)
    # Display table for immediate verification
    display(df_results.head())

In [ ]:
def plot_interactive_risk(df):
    # Create interactive scatter plot
    fig = px.scatter(
        df,
        x='Ann_Volatility',
        y='Max_Drawdown',
        hover_name='Path',  # Path number appears at the top of the hover box
        hover_data={
            'Ann_Volatility': ':.2%',
            'Max_Drawdown': ':.2%',
            'Cumulative_Return': ':.2%'
        },
        title='<b>Risk Dynamics: Volatility vs. Max Drawdown</b>',
        labels={
            'Ann_Volatility': 'Annualized Volatility (Risk)',
            'Max_Drawdown': 'Maximum Drawdown (Pain)'
        },
        template='plotly_white'
    )

    # Styling and Axis formatting
    fig.update_layout(
        title_font_size=20,
        xaxis_tickformat='.0%',
        yaxis_tickformat='.0%',
        hoverlabel=dict(bgcolor="white", font_size=14)
    )

    # Add a horizontal line at 0 for Drawdown reference
    fig.add_hline(y=0, line_dash="dash", line_color="gray")

    # Show the plot - Students can now hover to see Path IDs
    fig.show()

if 'df_results' in locals():
    plot_interactive_risk(df_results)

def summarize_risk_profiles(df):
    print("-" * 40)
    print("PORTFOLIO RISK SUMMARY (ALL PATHS)")
    print("-" * 40)

    # Calculate group-wide stats
    stats = pd.DataFrame({
        'Mean': [df['Ann_Volatility'].mean(), df['Max_Drawdown'].mean()],
        'Median': [df['Ann_Volatility'].median(), df['Max_Drawdown'].median()],
        'Min': [df['Ann_Volatility'].min(), df['Max_Drawdown'].min()],
        'Max': [df['Ann_Volatility'].max(), df['Max_Drawdown'].max()]
    }, index=['Volatility', 'Max Drawdown'])

    # Format for display
    display(stats.style.format("{:.2%}"))


if 'df_results' in locals():
    summarize_risk_profiles(df_results)

In [ ]:
def calculate_var_metrics(df, horizons=[1, 3, 6, 12, 24], confidence_level=0.95):
    var_results = []

    for h in horizons:
        # Calculate cumulative wealth for each path at month 'h'
        # we slice the dataframe up to month h and take the cumulative product
        wealth_at_h = (1 + df.iloc[:h, :]).prod() - 1

        # Calculate VaR (the 5th percentile for a 95% confidence level)
        # Note: VaR is usually expressed as a positive number representing a loss
        var_threshold = np.percentile(wealth_at_h, (1 - confidence_level) * 100)

        # Calculate Expected Shortfall (Average of all outcomes below the VaR threshold)
        expected_shortfall = wealth_at_h[wealth_at_h <= var_threshold].mean()

        var_results.append({
            'Horizon_Months': h,
            'VaR_95': var_threshold,
            'Expected_Shortfall': expected_shortfall,
            'Mean_Return': wealth_at_h.mean(),
            'Std_Dev': wealth_at_h.std()
        })

    return pd.DataFrame(var_results), wealth_at_h # returning wealth_at_h for plotting last horizon

# Run calculation
if df_paths is not None:
    df_var, _ = calculate_var_metrics(df_paths)

    # Export VaR results
    var_output_file = os.path.join(data_dir, f'L11_VaR_Analysis_{base_name}.csv')
    df_var.to_csv(var_output_file, index=False)

    print("-" * 30)
    print("VALUE AT RISK (VaR) & EXPECTED SHORTFALL")
    print("-" * 30)
    display(df_var.style.format({
        'VaR_95': '{:.2%}',
        'Expected_Shortfall': '{:.2%}',
        'Mean_Return': '{:.2%}',
        'Std_Dev': '{:.2%}'
    }))

In [ ]:
def plot_var_distributions(df, horizons=[1, 3, 6, 12, 24]):
    fig, axes = plt.subplots(len(horizons), 1, figsize=(10, 5 * len(horizons)))

    for i, h in enumerate(horizons):
        # Data for this horizon
        returns_at_h = (1 + df.iloc[:h, :]).prod() - 1

        # Calculate Stats
        mu, sigma = returns_at_h.mean(), returns_at_h.std()
        var_95 = np.percentile(returns_at_h, 5)

        # Plot Histogram
        sns.histplot(returns_at_h, kde=False, stat="density", ax=axes[i], color='lightgray', element="step")

        # Overlay Normal Curve
        x = np.linspace(returns_at_h.min(), returns_at_h.max(), 100)
        p = norm.pdf(x, mu, sigma)
        axes[i].plot(x, p, 'r', linewidth=2, label='Normal Distribution')

        # Highlight VaR Region
        axes[i].axvline(var_95, color='darkred', linestyle='--', lw=2, label=f'95% VaR: {var_95:.2%}')
        axes[i].fill_between(x, 0, norm.pdf(x, mu, sigma), where=(x <= var_95), color='red', alpha=0.3)

        # Formatting
        axes[i].set_title(f'Outcome Distribution: {h}-Month Horizon', fontweight='bold')
        axes[i].set_xlabel('Cumulative Return')
        axes[i].legend()
        axes[i].xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))

    plt.tight_layout()
    plt.savefig(os.path.join(base_dir, 'output', f'VaR_Distributions_{base_name}.png'))
    plt.show()

if df_paths is not None:
    plot_var_distributions(df_paths)